# Uni-Mol Fine-Tuning on LP-PDBBind

Fine-tunes **Uni-Mol v2** (84M params) on the LP-PDBBind dataset
(~19K protein-ligand complexes) for binding affinity prediction (pKa).

**Requirements:** Colab Pro with GPU runtime (A100 recommended).

## 1. Install Dependencies

In [ ]:
!pip install -q unimol_tools rdkit pandas scikit-learn matplotlib

## 2. Clone Repository & Download Data


In [ ]:
import os, urllib.request

!git clone https://github.com/handeboyaci/agentic-vlm.git
%cd agentic-vlm

# Download LP-PDBBind dataset (~19K protein-ligand complexes)
os.makedirs('data/pdbbind_deepchem/raw', exist_ok=True)
csv_path = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/THGLab/LP-PDBBind/master/dataset/LP_PDBBind.csv',
        csv_path
    )
    print(f'Downloaded LP_PDBBind.csv')
else:
    print('LP_PDBBind.csv already exists, skipping download.')


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = 'data/pdbbind_deepchem/raw/LP_PDBBind.csv'
df = pd.read_csv(CSV_PATH)
print(f'Loaded {len(df)} complexes')
print(f'Columns: {list(df.columns)}')
df.head(3)


In [ ]:
# Prepare data in Uni-Mol format: SMILES + TARGET
df_clean = df[['smiles', 'value', 'new_split']].copy()
df_clean = df_clean.rename(columns={'value': 'TARGET'})
df_clean = df_clean.dropna(subset=['smiles', 'TARGET'])

# Validate SMILES
from rdkit import Chem
valid_mask = df_clean['smiles'].apply(
    lambda s: Chem.MolFromSmiles(str(s)) is not None
)
df_clean = df_clean[valid_mask].reset_index(drop=True)

print(f'Valid molecules: {len(df_clean)}')
print(f'pKa range: {df_clean.TARGET.min():.1f} - {df_clean.TARGET.max():.1f}')
print(f'pKa mean: {df_clean.TARGET.mean():.2f} ± {df_clean.TARGET.std():.2f}')

df_clean['TARGET'].hist(bins=50, figsize=(8, 3))
import matplotlib.pyplot as plt
plt.xlabel('pKa'); plt.ylabel('Count'); plt.title('LP-PDBBind pKa Distribution')
plt.tight_layout(); plt.show()

In [ ]:
# Split using the dataset's native train/test split
train_df = df_clean[df_clean['new_split'] == 'train'][['smiles', 'TARGET']]
test_df = df_clean[df_clean['new_split'] == 'test'][['smiles', 'TARGET']]

# Rename for Uni-Mol: needs 'SMILES' (uppercase)
train_df = train_df.rename(columns={'smiles': 'SMILES'})
test_df = test_df.rename(columns={'smiles': 'SMILES'})

os.makedirs('data', exist_ok=True)
train_df.to_csv('data/unimol_train.csv', index=False)
test_df.to_csv('data/unimol_test.csv', index=False)

print(f'Train: {len(train_df)}, Test: {len(test_df)}')

## 3. Fine-Tune Uni-Mol v2

Training ~19K molecules for 50 epochs takes ~30 min on an A100.

In [ ]:
from unimol_tools import MolTrain

trainer = MolTrain(
    task='regression',
    data_type='molecule',
    epochs=50,
    batch_size=32,
    metrics='rmse',
    model_name='unimolv2',
    model_size='84m',
    save_path='./unimol_binding_model',
    remove_hs=False,
    target_cols='TARGET',
    learning_rate=1e-4,
)

trainer.fit(data='data/unimol_train.csv')

## 4. Evaluate on Test Set

In [ ]:
from unimol_tools import MolPredict
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

predictor = MolPredict(load_model='./unimol_binding_model')
predictions = predictor.predict(data='data/unimol_test.csv')

y_true = test_df['TARGET'].values
y_pred = predictions.flatten()

rmse = mean_squared_error(y_true, y_pred, squared=False)
r, _ = pearsonr(y_true, y_pred)

print(f'Uni-Mol Test RMSE:  {rmse:.3f}')
print(f'Uni-Mol Pearson r:  {r:.3f}')
print(f'\n(Compare to EGNN from-scratch RMSE for the benchmark table)')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true, y_pred, alpha=0.3, s=10, c='steelblue')
lims = [min(y_true.min(), y_pred.min()) - 0.5,
        max(y_true.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, 'r--', lw=1.5, label='Ideal')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('True pKa', fontsize=12)
ax.set_ylabel('Predicted pKa', fontsize=12)
ax.set_title(f'Uni-Mol v2 (84M) on LP-PDBBind\nRMSE={rmse:.3f}, r={r:.3f}',
             fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('unimol_scatter.png', dpi=150)
plt.show()

## 5. Export for Pipeline Integration

Download the `unimol_binding_model/` directory and place it in
your VLM project root. Then run:
```bash
python agent/pipeline.py --disease "Alzheimer's" --scoring unimol
```

In [ ]:
!zip -r unimol_binding_model.zip unimol_binding_model/

from google.colab import drive
import shutil

# Mount Drive and copy the zipped model over
drive.mount('/content/drive')
shutil.copy('unimol_binding_model.zip', '/content/drive/MyDrive/unimol_binding_model.zip')
print('Successfully saved unimol_binding_model.zip to Google Drive!')


## 6. Quick Inference Demo

In [ ]:
demo_smiles = [
    'CC(=O)OC1=CC=CC=C1C(=O)O',      # aspirin
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # caffeine
    'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O', # ibuprofen
    'c1ccc2c(c1)[nH]c1ccccc12',       # carbazole
]

demo_df = pd.DataFrame({'SMILES': demo_smiles})
demo_df.to_csv('data/demo_input.csv', index=False)

preds = predictor.predict(data='data/demo_input.csv')
print('\nInference results:')
for smi, pka in zip(demo_smiles, preds.flatten()):
    print(f'  {smi:55s} → pKa = {pka:.2f}')